In [ ]:
pip install transformers datasets torch

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from datasets import load_dataset

In [ ]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
race_dataset = load_dataset("race", "all")
train_data = race_dataset["train"]

In [ ]:
def preprocess_race_data(example):
    context = example["article"]
    question = example["question"]
    answer = example["answer"]
    options = " | ".join(example["options"])
    input_text = f"mcq question generation: Context: {context} Question: {question} Options: {options}"
    target_text = f"Answer: {answer}"
    return {"input_text": input_text, "target_text": target_text}

In [ ]:
processed_data = train_data.map(preprocess_race_data)

In [ ]:
inputs = processed_data["input_text"]
targets = processed_data["target_text"]

In [ ]:
max_input_length = 512
max_target_length = 64

def tokenize_data(input_texts, target_texts):
    inputs = tokenizer(input_texts, max_length=max_input_length, padding="max_length", truncation=True, return_tensors="pt")
    targets = tokenizer(target_texts, max_length=max_target_length, padding="max_length", truncation=True, return_tensors="pt")
    return inputs, targets

In [ ]:
input_encodings, target_encodings = tokenize_data(inputs, targets)

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch

class MCQDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels["input_ids"][idx])
        return item

dataset = MCQDataset(input_encodings, target_encodings)
data_loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
epochs = 5  # Run exactly 5 epochs
model.train()

for epoch in range(epochs):
    print(f"Starting epoch {epoch + 1}/{epochs}...")  # Print at the start of each epoch
    for step, batch in enumerate(data_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1} completed.")  # Print at the end of each epoch


In [ ]:
def generate_mcq(context):
    input_text = f"generate mcq: Context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    output_ids = model.generate(input_ids, max_length=128)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)